# Basic usage of LaTeX parser-interpreter **tables**

Anton Antonov   
March 2026

----

## Setup

In [ ]:
use experimental :rakuast;

use LaTeX::Grammar;
use CortexJS;

If the CortexJS commands below do not work evaluate the following magic cell (uncomment first):

In [ ]:
# #%bash
# npm install @cortex-js/compute-engine

---

## Small tables

Tabulate different to-formats:

In [ ]:
#%html
my @formulas = (
    '\\int _{0} ^{1} x^{2} d x',
    '\\sum_{n=1}^{10} n^2',
    '\\lim_{x\to0} \\frac{\\sin(x)}{x}',
);
my @targets = <AsciiMath MathJSON MathML WL>;

my @res = do for @formulas -> $fm {
   [Formula => $fm, |@targets.map({ $_ => latex-interpret($fm, actions => $_) })].Hash
}

#@res ==> to-html-table(field-names => ['Formula', |@targets])
@res.map(*<MathML>) ==> to-html-table()

In [ ]:
select-columns(@res, <Formula MathJSON WL>)
==> to-pretty-table(field-names => <Formula MathJSON WL>, align => 'l')

In [ ]:
#% html
[
    '4 x^2 + 12 x + 9',
    '4 * x^2 + 12 * x + 9',
    '\sqrt{4 * x^2 + 12 * x + 9}',
]
andthen $_.map({ %(expr => "latex«$_»", Cortex-JS => parse-latex($_).raku, 'LaTeX::Grammar' => latex-interpret($_).raku) })
andthen .&to-html(field-names => <expr Cortex-JS LaTeX::Grammar>, align => 'left')
andthen .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g)

----

## Larger table

In [ ]:
#%html

my @formulas = (
'\frac{-1214}{117}',
'\\sqrt{4 * x^2 + 12 * x + 9}',
'\\int_{0}^{1} x^{2} d x',
'\\sum_{n=1}^{10} n^2',
'\\lim_{x\\to0} \\frac{\\sin(x)}{x}',
'\\log_{5} x',
'\log\left( \frac{x+1}{x-1} \right)'
);
my @targets = <AsciiMath WL MathJSON>;

my @res = do for @formulas -> $fm {
    [Formula => $fm, MathML => "latex«$fm»", |@targets.map({ $_ => latex-interpret($fm, actions => $_).raku })].Hash
}

my $tbl = @res 
==> to-html(field-names => ['Formula', 'MathML', |@targets], align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { .subst(/ '"' | '&quot;' /, :g) }()

---

## "Neat" table by LLM

In [ ]:
#% chat html > markdown
Change this table into a table that has a column for the LaTeX formula, a column for the format (MathML, etc.), and the format value.
It should look like a neat table of subtables. Make sure all subtables have the same width.

TABLE:
```
<table border=1><thead><tr><th>Formula</th><th>MathML</th><th>AsciiMath</th><th>WL</th><th>MathJSON</th></tr></thead><tbody><tr><td align=left>\frac{-1214}{117}</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mfrac><mn>-1214</mn><mn>117</mn></mfrac></math></td><td align=left>-1214/117</td><td align=left>Rational[-1214,117]</td><td align=left>$[Rational, -1214, 117]</td></tr><tr><td align=left>\sqrt{4 * x^2 + 12 * x + 9}</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><msqrt><mrow><mrow><mrow><mn>4</mn><mo>&#xD7;</mo><msup><mi>x</mi><mn>2</mn></msup></mrow><mo>+</mo><mrow><mn>12</mn><mo>&#xD7;</mo><mi>x</mi></mrow></mrow><mo>+</mo><mn>9</mn></mrow></msqrt></math></td><td align=left>sqrt((4*x^2+12*x)+9)</td><td align=left>Sqrt[Plus[Plus[Times[4,Power[x,2]],Times[12,x]],9]]</td><td align=left>$[Root, [Add, [Add, [Multiply, 4, [Power, x, 2]], [Multiply, 12, x]], 9], 2]</td></tr><tr><td align=left>\int_{0}^{1} x^{2} d x</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mrow><msubsup><mo>&#x222B;</mo><mn>0</mn><mn>1</mn></msubsup><msup><mi>x</mi><mn>2</mn></msup><mrow><mo>d</mo><mi>x</mi></mrow></mrow></math></td><td align=left>int x^2</td><td align=left>Integrate[Power[x,2],\{x,0,1}]</td><td align=left>$[Integrate, [Function, [Block, [Power, x, 2]], x], [Limits, x, 0, 1]]</td></tr><tr><td align=left>\sum_{n=1}^{10} n^2</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mrow><msubsup><mo>&#x2211;</mo><mrow><mi>n</mi><mo>=</mo><mn>1</mn></mrow><mn>10</mn></msubsup><msup><mi>n</mi><mn>2</mn></msup></mrow></math></td><td align=left>sum n^2</td><td align=left>Sum[Power[n,2],\{n,1,10}]</td><td align=left>$[Sum, [Power, n, 2], [Limits, n, 1, 10]]</td></tr><tr><td align=left>\lim_{x\to0} \frac{\sin(x)}{x}</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mrow><munder><mi>lim</mi><mrow><mi>x</mi><mo>&#x2192;</mo><mn>0</mn></mrow></munder><mfrac><mrow><mi>sin</mi><mo>(</mo><mi>x</mi><mo>)</mo></mrow><mi>x</mi></mfrac></mrow></math></td><td align=left>lim_(x-&gt;0) sin(x)/x</td><td align=left>Limit[Times[ Sin[x] , Power[x, -1]],x-&gt;0]</td><td align=left>$[Limit, [Function, [Block, [Divide, [Sin, x], x]], x], 0]</td></tr><tr><td align=left>\log_{5} x</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mrow><msub><mi>log</mi><mn>5</mn></msub><mo>(</mo><mi>x</mi><mo>)</mo></mrow></math></td><td align=left>log_5(x)</td><td align=left>Log[5,x]</td><td align=left>$[Log, x, 5]</td></tr><tr><td align=left>\log\left( \frac{x+1}{x-1} \right)</td><td align=left><math xmlns=http://www.w3.org/1998/Math/MathML><mrow><mi>log</mi><mo>(</mo><mfrac><mrow><mi>x</mi><mo>+</mo><mn>1</mn></mrow><mrow><mi>x</mi><mo>-</mo><mn>1</mn></mrow></mfrac><mo>)</mo></mrow></math></td><td align=left>log((x+1)/(x-1))</td><td align=left>Log[Rational[Plus[x,1],Plus[x,Times[-1,1]]]]</td><td align=left>$[Log, [Divide, [Add, x, 1], [Subtract, x, 1]]]</td></tr></tbody></table>
```

<table border="1" width="100%">
  <thead>
    <tr>
      <th>LaTeX Formula</th>
      <th>Formats</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="left">\frac{-1214}{117}</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mfrac><mn>-1214</mn><mn>117</mn></mfrac></math></td></tr>
          <tr><td>AsciiMath</td><td>-1214/117</td></tr>
          <tr><td>WL</td><td>Rational[-1214,117]</td></tr>
          <tr><td>MathJSON</td><td>$[Rational, -1214, 117]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\sqrt{4 * x^2 + 12 * x + 9}</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><msqrt><mrow><mrow><mrow><mn>4</mn><mo>&#xD7;</mo><msup><mi>x</mi><mn>2</mn></msup></mrow><mo>+</mo><mrow><mn>12</mn><mo>&#xD7;</mo><mi>x</mi></mrow></mrow><mo>+</mo><mn>9</mn></mrow></msqrt></math></td></tr>
          <tr><td>AsciiMath</td><td>sqrt((4*x^2+12*x)+9)</td></tr>
          <tr><td>WL</td><td>Sqrt[Plus[Plus[Times[4,Power[x,2]],Times[12,x]],9]]</td></tr>
          <tr><td>MathJSON</td><td>$[Root, [Add, [Add, [Multiply, 4, [Power, x, 2]], [Multiply, 12, x]], 9], 2]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\int_{0}^{1} x^{2} d x</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mrow><msubsup><mo>&#x222B;</mo><mn>0</mn><mn>1</mn></msubsup><msup><mi>x</mi><mn>2</mn></msup><mrow><mo>d</mo><mi>x</mi></mrow></mrow></math></td></tr>
          <tr><td>AsciiMath</td><td>int x^2</td></tr>
          <tr><td>WL</td><td>Integrate[Power[x,2],{x,0,1}]</td></tr>
          <tr><td>MathJSON</td><td>$[Integrate, [Function, [Block, [Power, x, 2]], x], [Limits, x, 0, 1]]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\sum_{n=1}^{10} n^2</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mrow><msubsup><mo>&#x2211;</mo><mrow><mi>n</mi><mo>=</mo><mn>1</mn></mrow><mn>10</mn></msubsup><msup><mi>n</mi><mn>2</mn></msup></mrow></math></td></tr>
          <tr><td>AsciiMath</td><td>sum n^2</td></tr>
          <tr><td>WL</td><td>Sum[Power[n,2],{n,1,10}]</td></tr>
          <tr><td>MathJSON</td><td>$[Sum, [Power, n, 2], [Limits, n, 1, 10]]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\lim_{x\to0} \frac{\sin(x)}{x}</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mrow><munder><mi>lim</mi><mrow><mi>x</mi><mo>&#x2192;</mo><mn>0</mn></mrow></munder><mfrac><mrow><mi>sin</mi><mo>(</mo><mi>x</mi><mo>)</mo></mrow><mi>x</mi></mfrac></mrow></math></td></tr>
          <tr><td>AsciiMath</td><td>lim_(x-&gt;0) sin(x)/x</td></tr>
          <tr><td>WL</td><td>Limit[Times[ Sin[x] , Power[x, -1]],x-&gt;0]</td></tr>
          <tr><td>MathJSON</td><td>$[Limit, [Function, [Block, [Divide, [Sin, x], x]], x], 0]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\log_{5} x</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mrow><msub><mi>log</mi><mn>5</mn></msub><mo>(</mo><mi>x</mi><mo>)</mo></mrow></math></td></tr>
          <tr><td>AsciiMath</td><td>log_5(x)</td></tr>
          <tr><td>WL</td><td>Log[5,x]</td></tr>
          <tr><td>MathJSON</td><td>$[Log, x, 5]</td></tr>
        </table>
      </td>
    </tr>
    <tr>
      <td align="left">\log\left( \frac{x+1}{x-1} \right)</td>
      <td>
        <table border="1" width="100%">
          <tr><th>Format</th><th>Value</th></tr>
          <tr><td>MathML</td><td><math xmlns="http://www.w3.org/1998/Math/MathML"><mrow><mi>log</mi><mo>(</mo><mfrac><mrow><mi>x</mi><mo>+</mo><mn>1</mn></mrow><mrow><mi>x</mi><mo>-</mo><mn>1</mn></mrow></mfrac><mo>)</mo></mrow></math></td></tr>
          <tr><td>AsciiMath</td><td>log((x+1)/(x-1))</td></tr>
          <tr><td>WL</td><td>Log[Rational[Plus[x,1],Plus[x,Times[-1,1]]]]</td></tr>
          <tr><td>MathJSON</td><td>$[Log, [Divide, [Add, x, 1], [Subtract, x, 1]]]</td></tr>
        </table>
      </td>
    </tr>
  </tbody>
</table>

----

## Grouped

Just group by:

In [ ]:
#% html

@res 
==> { group-by(@res, 'Formula')}()
#==> { $_.map({ to-long-format($_.value) }) }()
#==> { .&to-html(field-names => <Variable Value>, align => 'left') }()
==> { .&to-html() }()

----

## Long format table

In [ ]:
#%html
@res ==> to-html()

In [ ]:
#% html

@res 
==> { .&to-long-format(id-columns => 'Formula', variables-to => 'Format') }()
==> { .sort(*<Formula Format>) }()
==> to-html(field-names => <Formula Format Value>, align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { $_.join("\n").subst(/ '"' | '&quot;' /, :g) }()

In [ ]:
#% html

@res 
==> { .&to-long-format(id-columns => 'Formula', variables-to => 'Format') }()
==> { group-by($_, 'Formula').map({ $_.value.sort(*<Formula Format>).kv.map(-> $i, %r { %r<Formula> ='' if $i; %r }) }) }()
==> { $_.flat(1) }()
==> to-html(field-names => <Formula Format Value>, align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { $_.join("\n").subst(/ '"' | '&quot;' /, :g) }()

In [ ]:
#%html

my @formulas = (
'\frac{-1214}{117}',
'\\sqrt{4 * x^2 + 12 * x + 9}',
'\\int_{0}^{1} x^{2} d x',
'\\sum_{n=1}^{10} n^2',
'\\lim_{x\\to0} \\frac{\\sin(x)}{x}',
'\\log_{5} x',
'\log\left( \frac{x+1}{x-1} \right)'
);
my @targets = <AsciiMath WL MathJSON>;

my @res = do for @formulas -> $fm {
    my %rec = [LaTeX => $fm, MathML => "latex«$fm»", |@targets.map({ $_ => latex-interpret($fm, actions => $_).raku })].Hash;
    my @rows = %rec.map({ %( Format => $_.key, '' => $_.value ) }).sort(*<Format>).Array;
    @rows[1,0,2,3,4]
}

@res 
==> { $_.map({ $_.&to-html(field-names => ['Format', ''], align => 'left') }) }()
==> { $_.map({ .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }) }()
==> { $_.join("\n").subst(/ '"' | '&quot;' /, :g) }()
==> { $_.subst('<th>', '<th align=left>', :g) }()

---

## Comparison with CortexJS LaTeX conversion to MathJSON

The LaTeX expressions can be converted to MathJSON using the Cortex-JS functions or by the Raku package ["LaTeX::Grammar"](https://raku.land/zef:antononcube/LaTeX::Grammar). Here is an example with the latter:

In [ ]:
#% html
my @res = do for @formulas -> $fm {
    [Formula => $fm, MathML => "latex«$fm»", 'LaTeX::Grammar' => latex-interpret($fm).raku, CortexJS => parse-latex($fm).raku].Hash
}

@res 
==> to-html(field-names => ['Formula', 'MathML', 'LaTeX::Grammar', 'CortexJS'], align => 'left')
==> { .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g) }()
==> { .subst('"', :g) }()